# Probability and Information Theory

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import seaborn as sns
from ipywidgets import Dropdown, FloatSlider, interact

### Concept 1: Expected Value, Variance, and Covariance Matrix


### What are They?

Imagine you test a model or observe tokens, sensor data, or user clicks:

1. **Expected Value ($\mathbb{E}[X]$, The Center / Mean):**
If you run the experiment a million times, what is the single average number you expect to see? It is the balance point of the data.
2. **Variance ($\text{Var}(X)$, The Spread / Uncertainty):**
How much does the data jump around its center?
* Low variance: All numbers clump tightly around the average (predictable).
* High variance: Numbers scatter wildly all over the place (noisy / risky).


3. **Covariance Matrix ($\Sigma$, The Relationship Grid):**
When you track **multiple features at once** (e.g., $X_0$ = sentence length, $X_1$ = sentiment score):
* Do they move together? If $X_0$ goes up, does $X_1$ also go up? (Positive covariance)
* Do they oppose each other? If $X_0$ goes up, does $X_1$ go down? (Negative covariance)
* Are they completely unrelated? (Zero covariance)



The **Covariance Matrix** is an $(n \times n)$ grid summarizing the variance of each feature along the diagonal, and the relationships between every pair of features in the off-diagonal cells.


### The Formulas

Given a dataset matrix $X$ of shape $(N \text{ samples} \times D \text{ features})$:

#### 1. Mean Vector ($\boldsymbol{\mu}$):

$$\boldsymbol{\mu} = \frac{1}{N} \sum_{i=1}^{N} \mathbf{x}_i$$

#### 2. Variance of feature $j$ ($\sigma_j^2$):

$$\text{Var}(X_j) = \frac{1}{N - 1} \sum_{i=1}^{N} (x_{i,j} - \mu_j)^2$$

#### 3. Covariance Matrix ($\Sigma$, Sigma):

First, center the data by subtracting the mean: $\tilde{X} = X - \boldsymbol{\mu}$.

Then compute:


$$\Sigma = \frac{1}{N - 1} \tilde{X}^\top \tilde{X}$$

#### How to read it out loud:

> *"The sample covariance matrix Sigma equals one over N minus one, times the centered data matrix transposed, multiplied by the centered data matrix."*

#### Breakdown of every piece:

* **$N$**: Number of data points / samples.
* **$N - 1$**: Bessel's correction (used in sample statistics to avoid underestimating true population variance).
* **$\tilde{X}$**: The zero-centered data matrix where the average of every column is $0$.
* **$\tilde{X}^\top \tilde{X}$**: Computes the dot product of every feature column against every other feature column!
* **Diagonal entries $\Sigma_{j,j}$**: The variance of feature $j$ with itself.
* **Off-diagonal entries $\Sigma_{j,k}$**: How feature $j$ and feature $k$ co-vary.


### Why AI Cares (In plain terms)

* **Batch Normalization & Layer Normalization:** Normalizing feature activations by subtracting the mean and dividing by the standard deviation ($\sqrt{\text{Variance}}$) is what makes training deep transformers and residual networks stable.
* **PCA & Dimensionality Reduction:** The principal components (eigenvectors from Linear Algebra) are calculated directly on this **Covariance Matrix**!
* **Variational Autoencoders (VAEs) & Diffusion Models:** Generative models explicitly predict mean vectors ($\boldsymbol{\mu}$) and variance/covariance matrices ($\Sigma$) to sample latent vectors from a Gaussian distribution.


### NumPy Implementation


In [5]:

# 1. Create synthetic data: 5 samples, each with 3 features
# Features could be: [reading_time, click_count, purchase_amount]
X = np.array([
    [10.0,  2.0,  50.0],
    [12.0,  3.0,  60.0],
    [9.0,   1.0,  45.0],
    [15.0,  5.0,  85.0],
    [11.0,  3.0,  55.0]
])

N = X.shape[0]  # 5 samples

# ==========================================
# Step 1: Compute Mean Vector (Expected Value)
# ==========================================
# Sum down the columns (axis=0) and divide by N
mean_manual = np.sum(X, axis=0) / N
mean_numpy = np.mean(X, axis=0)

# ==========================================
# Step 2: Center the Data (Subtract Mean)
# ==========================================
X_centered = X - mean_manual

# ==========================================
# Step 3: Compute Covariance Matrix Step-by-Step
# Formula: (X_centered.T @ X_centered) / (N - 1)
# ==========================================
cov_manual = (X_centered.T @ X_centered) / (N - 1)

# NumPy built-in (rowvar=False means each column is a feature)
cov_numpy = np.cov(X, rowvar=False)

# Extract individual variances from the diagonal
variances = np.diag(cov_manual)
# Standard deviation is sqrt(variance)
std_devs = np.sqrt(variances)

print("Original Data Shape:", X.shape)
print("\nMean Vector (Expected Value):\n", mean_manual)
print("\nCentered Data (Mean is 0):\n", X_centered)
print("\nCovariance Matrix (Manual):\n", np.round(cov_manual, 3))
print("\nCovariance Matrix (np.cov):\n", np.round(cov_numpy, 3))
print("\nFeature Variances (Diagonal):", np.round(variances, 3))
print("Feature Standard Deviations :", np.round(std_devs, 3))

# Notice: All off-diagonals are positive! 
# That means as reading_time increases, click_count and purchase_amount also increase.


Original Data Shape: (5, 3)

Mean Vector (Expected Value):
 [11.4  2.8 59. ]

Centered Data (Mean is 0):
 [[ -1.4  -0.8  -9. ]
 [  0.6   0.2   1. ]
 [ -2.4  -1.8 -14. ]
 [  3.6   2.2  26. ]
 [ -0.4   0.2  -4. ]]

Covariance Matrix (Manual):
 [[  5.3    3.35  35.5 ]
 [  3.35   2.2   22.25]
 [ 35.5   22.25 242.5 ]]

Covariance Matrix (np.cov):
 [[  5.3    3.35  35.5 ]
 [  3.35   2.2   22.25]
 [ 35.5   22.25 242.5 ]]

Feature Variances (Diagonal): [  5.3   2.2 242.5]
Feature Standard Deviations : [ 2.302  1.483 15.572]


In [6]:
# 1. Dataset
X = np.array([
    [10.0,  2.0,  50.0],
    [12.0,  3.0,  60.0],
    [9.0,   1.0,  45.0],
    [15.0,  5.0,  85.0],
    [11.0,  3.0,  55.0]
])

feature_names = ["Reading Time", "Click Count", "Purchase Amount"]
N = X.shape[0]

# Mean vector & centering
mean_vec = np.mean(X, axis=0)
X_centered = X - mean_vec

# Covariance matrix & Eigendecomposition (Axes of spread)
cov_mat = (X_centered.T @ X_centered) / (N - 1)
evals, evecs = np.linalg.eigh(cov_mat)

# Sort eigenvalues in descending order
idx = np.argsort(evals)[::-1]
evals = evals[idx]
evecs = evecs[:, idx]

# 2. Build 3D Point Cloud Scene
fig = go.Figure()

# Original centered data points
fig.add_trace(go.Scatter3d(
    x=X_centered[:, 0],
    y=X_centered[:, 1],
    z=X_centered[:, 2],
    mode="markers+text",
    marker={"size": 8, "color": "#1f77b4", "opacity": 0.9},
    text=[f"S{i+1}" for i in range(N)],
    textposition="top center",
    name="Centered Samples (X - μ)"
))

# Mean Origin Point (0, 0, 0)
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode="markers+text",
    marker={"size": 9, "color": "black", "symbol": "diamond"},
    text=["Mean (0,0,0)"],
    textposition="bottom center",
    name="Mean Center"
))

# 3. Draw Principal Variance Axes (Eigenvectors scaled by 2 * std dev)
colors = ["#d62728", "#2ca02c", "#ff7f0e"]
for i in range(3):
    scale = 2.0 * np.sqrt(evals[i])
    axis_end = evecs[:, i] * scale
    axis_start = -evecs[:, i] * scale
    
    fig.add_trace(go.Scatter3d(
        x=[axis_start[0], axis_end[0]],
        y=[axis_start[1], axis_end[1]],
        z=[axis_start[2], axis_end[2]],
        mode="lines",
        line={"color": colors[i], "width": 6},
        name=f"PC{i+1} Axis (Var = {evals[i]:.1f})"
    ))

# 4. Projection drop lines onto the floor (Reading Time vs Click Count)
z_min = float(np.min(X_centered[:, 2])) - 5.0
for pt in X_centered:
    fig.add_trace(go.Scatter3d(
        x=[pt[0], pt[0]],
        y=[pt[1], pt[1]],
        z=[z_min, pt[2]],
        mode="lines",
        line={"color": "gray", "dash": "dot", "width": 1.5},
        showlegend=False,
        hoverinfo="skip"
    ))

fig.add_trace(go.Scatter3d(
    x=X_centered[:, 0],
    y=X_centered[:, 1],
    z=np.full(N, z_min),
    mode="markers",
    marker={"size": 5, "color": "gray", "opacity": 0.5},
    name="Floor Shadow (x0 vs x1)"
))

fig.update_layout(
    title=f"3D Centered Data Cloud & Covariance Principal Spread Axes<br>"
          f"Dominant Variance along PC1: {evals[0]:.2f} ({evals[0]/np.sum(evals)*100:.1f}% total variance)",
    scene={
        "xaxis_title": f"Δ {feature_names[0]}",
        "yaxis_title": f"Δ {feature_names[1]}",
        "zaxis_title": f"Δ {feature_names[2]}",
        "aspectratio": {"x": 1, "y": 1, "z": 0.85},
        "camera": {"eye": {"x": -1.5, "y": -1.5, "z": 1.0}}
    },
    width=900,
    height=650,
    legend={"x": 0.02, "y": 0.95}
)

fig.show()

### Concept 2: Shannon Entropy (Measuring Information and Surprise)

#### What is Shannon Entropy?

Imagine someone tells you that the sun rose in the east this morning. That statement carries zero useful surprise; you already knew it was guaranteed to happen.

Now imagine someone tells you that it is snowing in the Sahara desert. That statement carries a massive amount of surprise because the event had an extremely low probability.

Claude Shannon turned this intuition into an exact mathematical measurement:

* **Information (Surprise) of a single event:** The rarer an event is, the more information it communicates when it occurs. If an event has probability $p$, its surprise is $\log_2(1 / p) = -\log_2(p)$.
* **Entropy ($H$):** The **average surprise** you expect across all possible outcomes of a system.
* If an outcome is 100% predictable (a coin with two heads), **Entropy = 0**. There is zero uncertainty.
* If all outcomes are equally likely (a fair coin: 50% heads, 50% tails), uncertainty is maximized, and **Entropy is at its highest**.



Entropy measures how chaotic, uncertain, or unpredictable a probability distribution is.

#### The Formula

For a discrete random variable $X$ with $n$ possible outcomes, each having probability $P(x_i)$:

$$H(X) = -\sum_{i=1}^{n} P(x_i) \log_2 P(x_i)$$

#### How to read it out loud:

> *"The entropy H of X equals the negative sum of P of x_i times log base two of P of x_i, summed across all outcomes from one to n."*

#### Breakdown of every piece:

* **$H(X)$**: The entropy score (measured in **bits** when using $\log_2$, or **nats** when using natural $\log$).
* **$P(x_i)$**: The probability of outcome $i$ occurring (must be between 0 and 1, and all probabilities must sum to 1).
* **$\log_2 P(x_i)$**: The logarithm of the probability. Since probabilities are $\le 1$, their logarithms are always negative numbers (or zero).
* **The negative sign ($-$) in front:** Cancels out the negative logarithm values so entropy is always a positive number or zero ($H \ge 0$).
* **Convention for $0 \log(0)$:** If an event has probability $0$, it never occurs, so $0 \log_2(0)$ is treated as $0$.

#### Why AI Cares:

* **Decision Trees (Information Gain):** Algorithms like Random Forests and XGBoost pick feature splits that cause the largest drop in entropy, sorting data into clean, predictable buckets.
* **LLM Sampling Temperature:** When an LLM generates tokens, a low temperature concentrates probabilities on the top token (low entropy, predictable text). A high temperature flattens probabilities across multiple tokens (high entropy, creative or chaotic text).
* **Foundation of Loss Functions:** Cross-entropy loss (the standard loss for almost all classifiers and language models) is built directly on top of Shannon entropy.

#### NumPy Implementation


In [7]:

# 1. Define three different probability distributions for a 4-sided die
# Distribution A: Completely uncertain (uniform distribution, maximum entropy)
p_uniform = np.array([0.25, 0.25, 0.25, 0.25])

# Distribution B: Biased / moderately predictable
p_biased = np.array([0.70, 0.10, 0.10, 0.10])

# Distribution C: 100% predictable (deterministic, minimum entropy)
p_certain = np.array([1.0, 0.0, 0.0, 0.0])

def compute_entropy(p, base=2):
    """
    Computes Shannon entropy of probability distribution p.
    base=2 outputs entropy in 'bits'.
    base=np.e outputs entropy in 'nats'.
    """
    # Filter out zero probabilities to avoid log2(0) runtime warnings
    p_nonzero = p[p > 0]
    
    if base == 2:
        log_p = np.log2(p_nonzero)
    else:
        log_p = np.log(p_nonzero)
        
    # Formula: -sum(p * log(p))
    entropy = -np.sum(p_nonzero * log_p)
    return entropy

entropy_uniform = compute_entropy(p_uniform)
entropy_biased = compute_entropy(p_biased)
entropy_certain = compute_entropy(p_certain)

print("Distribution A (Equal 25% each) :", p_uniform)
print(f"Entropy A (Maximum uncertainty): {entropy_uniform:.4f} bits\n")

print("Distribution B (70% on one face):", p_biased)
print(f"Entropy B (Moderate uncertainty): {entropy_biased:.4f} bits\n")

print("Distribution C (100% certain)   :", p_certain)
print(f"Entropy C (Zero uncertainty)    : {entropy_certain:.4f} bits")

Distribution A (Equal 25% each) : [0.25 0.25 0.25 0.25]
Entropy A (Maximum uncertainty): 2.0000 bits

Distribution B (70% on one face): [0.7 0.1 0.1 0.1]
Entropy B (Moderate uncertainty): 1.3568 bits

Distribution C (100% certain)   : [1. 0. 0. 0.]
Entropy C (Zero uncertainty)    : -0.0000 bits


In [ ]:
sns.set_theme(style="whitegrid")
palette = sns.color_palette("deep")

# 1. Three dice distributions
dists = {
    "Uniform A (Max Uncertainty)": np.array([0.25, 0.25, 0.25, 0.25]),
    "Biased B (Moderate)": np.array([0.70, 0.10, 0.10, 0.10]),
    "Certain C (Zero Surprise)": np.array([1.0, 0.0, 0.0, 0.0])
}

def get_entropy_bits(p):
    p_nz = p[p > 0]
    return -np.sum(p_nz * np.log2(p_nz))

@interact(
    p_heads=FloatSlider(
        min=0.01, max=0.99, step=0.02, value=0.5,
        description="P(Heads):",
        continuous_update=True
    )
)
def visualize_shannon_entropy(p_heads=0.5):
    _fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [1.2, 1]})

    # -------------------------------------------------------------
    # Panel 1: 4-Sided Die Distributions & Their Total Entropy
    # -------------------------------------------------------------
    x_faces = np.arange(1, 5)
    width = 0.26
    colors = [palette[0], palette[1], palette[2]]

    for idx, (label, p_arr) in enumerate(dists.items()):
        h_val = get_entropy_bits(p_arr)
        bars = axes[0].bar(
            x_faces + (idx - 1) * width, p_arr, width=width,
            label=f"{label}: H = {h_val:.2f} bits",
            color=colors[idx], alpha=0.85, edgecolor="black"
        )
        
        # Annotate probabilities on bars
        for b in bars:
            h = b.get_height()
            if h > 0.05:
                axes[0].text(
                    b.get_x() + b.get_width() / 2, h + 0.02,
                    f"{h:.2f}", ha="center", va="bottom", fontsize=8.5, weight="bold"
                )

    axes[0].set_xticks(x_faces)
    axes[0].set_xticklabels([f"Face {i}" for i in x_faces], fontsize=10, weight="bold")
    axes[0].set_ylim(0, 1.15)
    axes[0].set_ylabel("Probability P(X = x)", fontsize=11)
    axes[0].set_title(
        r"4-Sided Die: $H(X) = -\sum p_i \log_2(p_i)$" + "\n" +
        r"Uniform has max theoretical bits ($\log_2(4) = 2.0$ bits)",
        fontsize=11, weight="bold"
    )
    axes[0].legend(loc="upper right", frameon=True, fontsize=9)

    # -------------------------------------------------------------
    # Panel 2: Continuous Binary Entropy Curve H(p) = -p log2 p - (1-p) log2 (1-p)
    # -------------------------------------------------------------
    p_curve = np.linspace(0.001, 0.999, 300)
    h_curve = -(p_curve * np.log2(p_curve) + (1.0 - p_curve) * np.log2(1.0 - p_curve))
    
    # Current slider value
    current_h = -(p_heads * np.log2(p_heads) + (1.0 - p_heads) * np.log2(1.0 - p_heads))

    axes[1].plot(p_curve, h_curve, color="black", linewidth=2.5, label="Binary Entropy Function")
    axes[1].scatter([p_heads], [current_h], color="#d62728", s=100, zorder=6)
    axes[1].axvline(p_heads, color="#d62728", linestyle=":", alpha=0.7)
    axes[1].axhline(current_h, color="#d62728", linestyle=":", alpha=0.7)

    # Annotate current point
    axes[1].annotate(
        f"p = {p_heads:.2f}\nH = {current_h:.3f} bits",
        (p_heads, current_h),
        textcoords="offset points", xytext=(15, -15),
        fontsize=10, weight="bold", color="#d62728",
        bbox={"boxstyle": "round,pad=0.3", "fc": "white", "ec": "#d62728", "lw": 1.2}
    )

    axes[1].set_xlim(0, 1)
    axes[1].set_ylim(0, 1.1)
    axes[1].set_xlabel("Probability of Event p", fontsize=11)
    axes[1].set_ylabel("Entropy H(p) in Bits", fontsize=11)
    axes[1].set_title(
        r"Coin Flip Entropy Curve" + "\n" +
        r"Peaks at $p=0.5$ (Fair coin = 1.0 bit) | Drops to 0 at certainty",
        fontsize=11, weight="bold"
    )
    axes[1].legend(loc="lower center", frameon=True, fontsize=9)

    plt.tight_layout()
    plt.show()

interactive(children=(FloatSlider(value=0.5, description='P(Heads):', max=0.99, min=0.01, step=0.02), Output()…

### Concept 3: Cross-Entropy Loss

#### What is Cross-Entropy Loss?

In Concept 2, Shannon Entropy measured how much surprise exists inside a single distribution $P$.

In supervised learning, you always have two distributions:

1. **The True Target ($P$):** The ground truth reality (e.g., this image is 100% a dog: $P = [1.0, 0.0]$).
2. **The Model's Prediction ($Q$):** What the neural network currently guesses after looking at the inputs (e.g., $Q = [0.8, 0.2]$).

**Cross-Entropy answers one question:**

> *"If the real world behaves according to distribution $P$, but our model uses distribution $Q$ to make its predictions, how much total surprise / penalty do we incur?"*

* When the model's predicted probabilities $Q$ match the ground truth $P$ perfectly, cross-entropy hits its lowest possible value (equal to the entropy of $P$).
* If the model assigns a very low probability to the true class (e.g., the ground truth is "dog", but the model only assigned $0.01$ probability to "dog"), $-\log(0.01)$ blows up to a large number, applying a heavy penalty.

#### The Formula

For a classification problem with $C$ possible classes:

$$H(P, Q) = -\sum_{i=1}^{C} P(y_i) \log(Q(y_i))$$

For a single training example where the ground truth is a one-hot vector (meaning class $k$ is $1.0$ and all other classes are $0.0$), the sum collapses down to a single term:

$$L_{\text{CE}} = -\log(Q(y_k))$$

#### How to read it out loud:

> *"Cross-entropy of P and Q equals the negative sum of the true probability of y_i times the natural log of the predicted probability of y_i across all classes from one to C."*

#### Breakdown of every piece:

* **$P(y_i)$**: The true ground truth probability for class $i$ (usually 0 or 1).
* **$Q(y_i)$**: The model's predicted probability for class $i$ (outputs from a Softmax layer, between 0 and 1).
* **$\log$**: The natural logarithm ($\ln$).
* **$-\log(Q(y_k))$**: The negative log-likelihood of the correct class. If $Q(y_k) = 1.0$, loss is $-\log(1.0) = 0$. If $Q(y_k) \to 0$, loss $\to \infty$.

#### Why AI Cares:

* **Standard Loss for Classification:** Used everywhere across computer vision (image classification), NLP (sentiment analysis, intent detection), and recommendation systems.
* **Autoregressive Language Modeling:** Every next-token prediction step in an LLM (predicting token $t+1$ from a vocabulary of 32,000 to 128,000 words) evaluates cross-entropy loss against the single actual target token.

#### NumPy Implementation

In [9]:

# Number of classes = 3 (e.g., [Cat, Dog, Bird])
# True label: The image is a Dog (Index 1)
# One-hot encoded ground truth distribution P
p_true = np.array(object=[0.0, 1.0, 0.0])

# Scenario 1: Model is confident and correct (High probability on Dog)
q_good = np.array([0.05, 0.90, 0.05])

# Scenario 2: Model is confused / uncertain
q_confused = np.array([0.33, 0.34, 0.33])

# Scenario 3: Model is confident and completely WRONG (Thinks it's a Cat)
q_wrong = np.array([0.90, 0.02, 0.08])

def compute_cross_entropy(p, q, epsilon=1e-15):
    """
    Computes cross-entropy loss between ground truth p and prediction q.
    epsilon prevents log(0) which results in -infinity.
    """
    # Clip q to prevent numerical crash if a probability is exactly 0.0
    q_clipped = np.clip(q, epsilon, 1.0 - epsilon)
    
    # Formula: -sum(p * log(q))
    loss = -np.sum(p * np.log(q_clipped))
    return loss

loss_good = compute_cross_entropy(p_true, q_good)
loss_confused = compute_cross_entropy(p_true, q_confused)
loss_wrong = compute_cross_entropy(p_true, q_wrong)

print(f"Ground Truth Distribution P: {p_true} (True class is Dog)\n")
print(f"Prediction 1 (Confident & Correct): {q_good}")
print(f"Cross-Entropy Loss 1             : {loss_good:.4f}\n")

print(f"Prediction 2 (Unsure / Confused)  : {q_confused}")
print(f"Cross-Entropy Loss 2             : {loss_confused:.4f}\n")

print(f"Prediction 3 (Confident & WRONG)  : {q_wrong}")
print(f"Cross-Entropy Loss 3             : {loss_wrong:.4f}")

Ground Truth Distribution P: [0. 1. 0.] (True class is Dog)

Prediction 1 (Confident & Correct): [0.05 0.9  0.05]
Cross-Entropy Loss 1             : 0.1054

Prediction 2 (Unsure / Confused)  : [0.33 0.34 0.33]
Cross-Entropy Loss 2             : 1.0788

Prediction 3 (Confident & WRONG)  : [0.9  0.02 0.08]
Cross-Entropy Loss 3             : 3.9120


In [10]:
sns.set_theme(style="whitegrid")
palette = sns.color_palette("deep")

classes = ["Cat", "Dog", "Bird"]
p_true = np.array([0.0, 1.0, 0.0])  # True class: Dog (Index 1)

scenarios = {
    "Scenario 1: Confident & Correct": np.array([0.05, 0.90, 0.05]),
    "Scenario 2: Unsure / Confused": np.array([0.33, 0.34, 0.33]),
    "Scenario 3: Confident & WRONG": np.array([0.90, 0.02, 0.08])
}

def compute_ce(p, q, eps=1e-15):
    q_c = np.clip(q, eps, 1.0 - eps)
    return -np.sum(p * np.log(q_c))

@interact(
    selected=Dropdown(
        options=list(scenarios.keys()),
        value="Scenario 1: Confident & Correct",
        description="Scenario:"
    )
)
def visualize_cross_entropy(selected):
    q_pred = scenarios[selected]
    loss_val = compute_ce(p_true, q_pred)
    q_dog = q_pred[1]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [1.1, 1]})

    # -------------------------------------------------------------
    # Panel 1: Target vs Predicted Probabilities
    # -------------------------------------------------------------
    x_pos = np.arange(len(classes))
    bar_width = 0.35

    bars_true = axes[0].bar(
        x_pos - bar_width / 2, p_true, width=bar_width,
        color="black", alpha=0.8, label="Ground Truth P (Target = Dog)"
    )
    
    bar_color = "#2ca02c" if "Correct" in selected else ("#ff7f0e" if "Unsure" in selected else "#d62728")
    bars_pred = axes[0].bar(
        x_pos + bar_width / 2, q_pred, width=bar_width,
        color=bar_color, alpha=0.85, label=f"Predicted Q (Loss = {loss_val:.4f})"
    )

    for b in bars_true:
        h = b.get_height()
        axes[0].text(b.get_x() + b.get_width() / 2, h + 0.02, f"{h:.2f}", ha="center", fontsize=9, weight="bold")
    for b in bars_pred:
        h = b.get_height()
        axes[0].text(b.get_x() + b.get_width() / 2, h + 0.02, f"{h:.2f}", ha="center", fontsize=9, weight="bold", color=bar_color)

    axes[0].set_xticks(x_pos)
    axes[0].set_xticklabels(classes, fontsize=11, weight="bold")
    axes[0].set_ylim(0, 1.2)
    axes[0].set_ylabel("Probability", fontsize=11)
    axes[0].set_title("Probability Distributions: True P vs Predicted Q", fontsize=11, weight="bold")
    axes[0].legend(loc="upper right", frameon=True, fontsize=9)

    # -------------------------------------------------------------
    # Panel 2: The Negative Log Loss Curve: L = -ln(q_true)
    # -------------------------------------------------------------
    qs = np.linspace(0.01, 1.0, 300)
    losses = -np.log(qs)

    axes[1].plot(qs, losses, color="black", linewidth=2.5, label=r"Loss Curve: $L = -\ln(q_{\mathrm{true}})$")
    
    axes[1].scatter([q_dog], [loss_val], color=bar_color, s=110, zorder=6)
    axes[1].axvline(q_dog, color=bar_color, linestyle=":", alpha=0.7)
    axes[1].axhline(loss_val, color=bar_color, linestyle=":", alpha=0.7)

    axes[1].annotate(
        rf"$q_{{\mathrm{{dog}}}} = {q_dog:.2f}$" + "\n" + rf"$\mathrm{{Loss}} = {loss_val:.4f}$",
        (q_dog, loss_val),
        textcoords="offset points",
        xytext=(15 if q_dog < 0.6 else -110, 15),
        fontsize=10, weight="bold", color=bar_color,
        bbox={"boxstyle": "round,pad=0.3", "fc": "white", "ec": bar_color, "lw": 1.5}
    )

    axes[1].set_xlim(0, 1.02)
    axes[1].set_ylim(0, 4.8)
    axes[1].set_xlabel(r"Predicted Confidence on True Class ($q_{\mathrm{dog}}$)", fontsize=11)
    axes[1].set_ylabel("Cross-Entropy Loss", fontsize=11)
    axes[1].set_title(r"Penalty Growth as $q_{\mathrm{true}} \to 0$", fontsize=11, weight="bold")
    axes[1].legend(loc="upper right", frameon=True, fontsize=9)

    fig.suptitle(
        rf"Cross-Entropy: $-\sum p_i \ln(q_i) = -\ln(q_{{\mathrm{{dog}}}})$  |  Selected: {selected}",
        fontsize=12, weight="bold", y=1.03
    )

    plt.tight_layout()
    plt.show()

interactive(children=(Dropdown(description='Scenario:', options=('Scenario 1: Confident & Correct', 'Scenario …

### Concept 4: Kullback-Leibler (KL) Divergence

#### What is KL Divergence?

In Concept 2, you learned **Entropy** (uncertainty inside one distribution).
In Concept 3, you learned **Cross-Entropy** (total penalty of using $Q$ when the true distribution is $P$).

**KL Divergence answers this question:**

> *"How much **extra** surprise or information loss occurs specifically because we approximated the true distribution $P$ with our model distribution $Q$, rather than using $P$ directly?"*

Think of it as the **relative entropy** or an information-theoretic distance between two distributions:

* If $Q$ is identical to $P$, the divergence is **$0$** (no information lost).
* The more $Q$ deviates from $P$, the larger the score becomes.
* **Important quirk:** It is **not** a symmetric distance metric. The divergence from $P$ to $Q$ is usually different from the divergence from $Q$ to $P$:

$$D_{\text{KL}}(P \parallel Q) \neq D_{\text{KL}}(Q \parallel P)$$



The elegant mathematical link connecting all three concepts:


$$\text{Cross-Entropy}(P, Q) = \text{Entropy}(P) + D_{\text{KL}}(P \parallel Q)$$

Since the true distribution $P$ is fixed (so its internal Entropy $(P)$ is constant), **minimizing cross-entropy loss is mathematically identical to minimizing KL divergence**.

#### The Formula

For two discrete probability distributions $P$ and $Q$ over $n$ events:

$$D_{\text{KL}}(P \parallel Q) = \sum_{i=1}^{n} P(x_i) \log\left(\frac{P(x_i)}{Q(x_i)}\right)$$

Using logarithmic division rules ($\log(a/b) = \log(a) - \log(b)$), it expands to:

$$D_{\text{KL}}(P \parallel Q) = \sum_{i=1}^{n} P(x_i) \log(P(x_i)) - \sum_{i=1}^{n} P(x_i) \log(Q(x_i))$$

$$D_{\text{KL}}(P \parallel Q) = -H(P) + H(P, Q)$$

#### How to read it out loud:

> *"The KL divergence from Q to P (written D-KL of P parallel Q) equals the sum of P of x_i times the natural log of P of x_i divided by Q of x_i across all outcomes from one to n."*

#### Breakdown of every piece:

* **$D_{\text{KL}}(P \parallel Q)$**: The divergence score (always $\ge 0$ by Gibbs' inequality).
* **$P(x_i)$**: The true reference probability for outcome $i$.
* **$Q(x_i)$**: The approximating / predicted probability for outcome $i$.
* **$\frac{P(x_i)}{Q(x_i)}$**: The probability ratio. If the model prediction matches the real probability ($Q = P$), the ratio is $1$, and $\log(1) = 0$.
* **Asymmetry note:** If $P(x_i) > 0$ but $Q(x_i) \to 0$, the ratio blows up to infinity. This makes KL divergence penalize models severely if they underestimate the probability of events that actually occur.

#### Why AI Cares:

* **Variational Autoencoders (VAEs):** The VAE loss has two terms: reconstruction loss plus a **KL divergence penalty** that forces the learned latent space distribution to match a standard Gaussian bell curve $\mathcal{N}(0, I)$.
* **Reinforcement Learning from Human Feedback (RLHF / PPO):** When fine-tuning LLMs with RLHF, a KL divergence penalty prevents the policy model from drifting too far away from the initial base reference model, stopping model collapse and reward hacking.
* **Knowledge Distillation:** When compressing a massive teacher model into a smaller student model, the student is trained by minimizing the KL divergence between its output distribution and the teacher's soft probability distribution.

#### NumPy Implementation


In [11]:

# Define true reference distribution P (e.g., ground truth token distribution)
P = np.array([0.40, 0.35, 0.15, 0.10])

# Model Q1: Very close approximation to P
Q1 = np.array([0.38, 0.36, 0.16, 0.10])

# Model Q2: Poor approximation (drastically different spread)
Q2 = np.array([0.10, 0.10, 0.40, 0.40])

def compute_kl_divergence(p, q, epsilon=1e-12):
    """
    Computes KL Divergence D_KL(P || Q) using natural logarithm.
    """
    # Prevent divide-by-zero or log(0)
    p = np.clip(p, epsilon, 1.0)
    q = np.clip(q, epsilon, 1.0)
    
    # Formula: sum(P * log(P / Q))
    kl = np.sum(p * np.log(p / q))
    return kl

kl_q1 = compute_kl_divergence(P, Q1)
kl_q2 = compute_kl_divergence(P, Q2)

# Demonstrate asymmetry: D_KL(P || Q1) vs D_KL(Q1 || P)
kl_reverse = compute_kl_divergence(Q1, P)

# Verify the fundamental identity: Cross-Entropy = Entropy(P) + KL(P || Q)
entropy_p = -np.sum(P * np.log(P))
cross_entropy_q1 = -np.sum(P * np.log(Q1))
reconstructed_ce = entropy_p + kl_q1

print(f"True Distribution P: {P}\n")
print(f"Close Prediction Q1 : {Q1}")
print(f"KL Divergence D_KL(P || Q1): {kl_q1:.6f} nats\n")

print(f"Distant Prediction Q2: {Q2}")
print(f"KL Divergence D_KL(P || Q2): {kl_q2:.6f} nats\n")

print("Asymmetry Check:")
print(f"D_KL(P || Q1) = {kl_q1:.6f}")
print(f"D_KL(Q1 || P) = {kl_reverse:.6f} (Different value!)\n")

print("Mathematical Identity Check:")
print(f"True Entropy H(P)         : {entropy_p:.6f}")
print(f"Cross-Entropy H(P, Q1)    : {cross_entropy_q1:.6f}")
print(f"H(P) + D_KL(P || Q1)      : {reconstructed_ce:.6f}")
print(f"Are they identical?       : {np.isclose(cross_entropy_q1, reconstructed_ce)}")


True Distribution P: [0.4  0.35 0.15 0.1 ]

Close Prediction Q1 : [0.38 0.36 0.16 0.1 ]
KL Divergence D_KL(P || Q1): 0.000977 nats

Distant Prediction Q2: [0.1 0.1 0.4 0.4]
KL Divergence D_KL(P || Q2): 0.707231 nats

Asymmetry Check:
D_KL(P || Q1) = 0.000977
D_KL(Q1 || P) = 0.000976 (Different value!)

Mathematical Identity Check:
True Entropy H(P)         : 1.248781
Cross-Entropy H(P, Q1)    : 1.249757
H(P) + D_KL(P || Q1)      : 1.249757
Are they identical?       : True


In [12]:
sns.set_theme(style="whitegrid")
palette = sns.color_palette("deep")

# Data definitions
tokens = ["Token A", "Token B", "Token C", "Token D"]
P = np.array([0.40, 0.35, 0.15, 0.10])

models = {
    "Model Q1 (Close Approximation)": np.array([0.38, 0.36, 0.16, 0.10]),
    "Model Q2 (Poor / Distant Approximation)": np.array([0.10, 0.10, 0.40, 0.40])
}

def compute_kl(p, q, eps=1e-12):
    p = np.clip(p, eps, 1.0)
    q = np.clip(q, eps, 1.0)
    return np.sum(p * np.log(p / q))

@interact(
    model_choice=Dropdown(
        options=list(models.keys()),
        value="Model Q1 (Close Approximation)",
        description="Model Q:"
    )
)
def plot_kl_divergence(model_choice):
    Q = models[model_choice]
    
    # Per-element divergence contribution: P_i * ln(P_i / Q_i)
    per_token_kl = P * np.log(P / Q)
    total_kl_forward = np.sum(per_token_kl)
    total_kl_reverse = compute_kl(Q, P)
    
    # Identity values
    entropy_P = -np.sum(P * np.log(P))
    cross_entropy = -np.sum(P * np.log(Q))

    fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [1.2, 1]})

    # -------------------------------------------------------------
    # Panel 1: Probability Spread & Element-Wise KL Contributions
    # -------------------------------------------------------------
    x_pos = np.arange(len(tokens))
    bar_width = 0.28

    axes[0].bar(
        x_pos - bar_width, P, width=bar_width,
        color="black", alpha=0.8, label="Ground Truth P"
    )
    
    model_color = palette[0] if "Q1" in model_choice else palette[3]
    axes[0].bar(
        x_pos, Q, width=bar_width,
        color=model_color, alpha=0.85, label=f"Model {model_choice.split()[1]}"
    )

    # Element-wise KL bar
    axes[0].bar(
        x_pos + bar_width, per_token_kl, width=bar_width,
        color=palette[2], alpha=0.9, label=r"Token Penalty: $P_i \ln(P_i / Q_i)$"
    )

    # Annotations
    for i in range(len(tokens)):
        axes[0].text(x_pos[i] - bar_width, P[i] + 0.015, f"{P[i]:.2f}", ha="center", fontsize=8.5, weight="bold")
        axes[0].text(x_pos[i], Q[i] + 0.015, f"{Q[i]:.2f}", ha="center", fontsize=8.5, weight="bold", color=model_color)
        axes[0].text(
            x_pos[i] + bar_width, 
            per_token_kl[i] + (0.015 if per_token_kl[i] >= 0 else -0.04), 
            f"{per_token_kl[i]:+.3f}", 
            ha="center", fontsize=8.5, weight="bold", color=palette[2]
        )

    axes[0].axhline(0, color="gray", linewidth=0.8, linestyle="--")
    axes[0].set_xticks(x_pos)
    axes[0].set_xticklabels(tokens, fontsize=11, weight="bold")
    axes[0].set_ylabel("Probability / Nat Contribution", fontsize=11)
    axes[0].set_title(
        r"Distributions & Per-Token Divergence: $D_{\mathrm{KL}} = " + f"{total_kl_forward:.4f}$ nats",
        fontsize=11, weight="bold"
    )
    axes[0].legend(loc="upper right", frameon=True, fontsize=9)

    # -------------------------------------------------------------
    # Panel 2: Stacked Decomposition & Asymmetry Check
    # -------------------------------------------------------------
    # Stacked bar showing Cross Entropy = H(P) + D_KL(P || Q)
    axes[1].bar(
        [0], [entropy_P], width=0.45,
        color=palette[7], label=rf"True Entropy $H(P) = {entropy_P:.3f}$"
    )
    axes[1].bar(
        [0], [total_kl_forward], width=0.45, bottom=[entropy_P],
        color=palette[2], label=rf"Excess Loss $D_{{\mathrm{{KL}}}}(P \parallel Q) = {total_kl_forward:.3f}$"
    )

    # Asymmetry Comparison Bars
    axes[1].bar([1.1], [total_kl_forward], width=0.35, color=palette[2], label=rf"$D_{{\mathrm{{KL}}}}(P \parallel Q) = {total_kl_forward:.3f}$")
    axes[1].bar([1.6], [total_kl_reverse], width=0.35, color=palette[4], label=rf"$D_{{\mathrm{{KL}}}}(Q \parallel P) = {total_kl_reverse:.3f}$")

    # Annotate total cross entropy
    axes[1].text(0, cross_entropy + 0.05, f"CE: {cross_entropy:.3f}", ha="center", fontsize=9.5, weight="bold")
    axes[1].text(1.1, total_kl_forward + 0.05, f"{total_kl_forward:.3f}", ha="center", fontsize=9, weight="bold", color=palette[2])
    axes[1].text(1.6, total_kl_reverse + 0.05, f"{total_kl_reverse:.3f}", ha="center", fontsize=9, weight="bold", color=palette[4])

    axes[1].set_xticks([0, 1.35])
    axes[1].set_xticklabels(
        [r"Cross-Entropy Split" + "\n" + r"$H(P) + D_{\mathrm{KL}}$", r"Asymmetry" + "\n" + r"$P \parallel Q$ vs $Q \parallel P$"],
        fontsize=10, weight="bold"
    )
    axes[1].set_ylabel("Nats", fontsize=11)
    axes[1].set_title(
        r"Loss Identity & Asymmetry Check",
        fontsize=11, weight="bold"
    )
    axes[1].legend(loc="upper left", frameon=True, fontsize=8.5)

    fig.suptitle(
        rf"Kullback-Leibler Divergence Analysis: {model_choice}",
        fontsize=12, weight="bold", y=1.03
    )

    plt.tight_layout()
    plt.show()

interactive(children=(Dropdown(description='Model Q:', options=('Model Q1 (Close Approximation)', 'Model Q2 (P…

### Concept 5: Softmax and the Numerically Stable Log-Sum-Exp Trick

#### What is Softmax and Log-Sum-Exp?

Neural network linear heads output unconstrained real numbers called **logits** ($z$), which can be negative, zero, or arbitrarily large positive values (e.g., $[-10.5, 0.2, 850.0]$).

To turn these raw logits into valid probabilities that:

1. Are strictly positive ($0 < P(y_i) < 1$)
2. Sum up to exactly $1.0$ ($100\%$)

we use the **Softmax function**. It exponentiates every logit ($e^{z_i}$) to force it positive, then divides by the sum of all exponentiated logits.

**The Floating-Point Catastrophe:**
In computers, 64-bit and 32-bit floating-point numbers have strict limits:

* In standard 32-bit floats (`float32`), $e^{88} \approx 1.6 \times 10^{38}$. If a logit is larger than $88.7$, $e^z$ overflows to `inf` (infinity).
* When you divide `inf / inf`, the computer produces `NaN` (Not a Number), causing model loss to crash.
* Conversely, if logits are very negative (e.g., $-1000$), $e^{-1000}$ underflows to exact $0.0$, leading to $\log(0) = -\infty$.

The **Log-Sum-Exp (LSE) Trick** is a mathematical identity that guarantees **numerical stability**:
By subtracting the maximum value ($c = \max(\mathbf{z})$) from every logit before exponentiating, the largest value becomes $e^0 = 1.0$. This prevents numbers from exceeding $1.0$ during intermediate exponentiation, making numerical overflow impossible without changing the final output probabilities.

#### The Formulas

**1. Standard Softmax:**

$$\text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{C} e^{z_j}}$$

**2. Mathematically Equivalent Numerically Stable Softmax:**
Let $c = \max(\mathbf{z})$:

$$\text{Softmax}(z_i) = \frac{e^{z_i - c}}{\sum_{j=1}^{C} e^{z_j - c}}$$

*Proof of equivalence:*


$$\frac{e^{z_i - c}}{\sum_{j} e^{z_j - c}} = \frac{e^{z_i} \cdot e^{-c}}{\sum_{j} (e^{z_j} \cdot e^{-c})} = \frac{e^{z_i} e^{-c}}{e^{-c} \sum_{j} e^{z_j}} = \frac{e^{z_i}}{\sum_{j} e^{z_j}}$$

**3. The Log-Sum-Exp (LSE) Formula:**
Used to compute the log-denominator $\log \sum e^{z_j}$ safely when evaluating Log-Softmax or Cross-Entropy:

$$\text{LSE}(\mathbf{z}) = c + \log\left(\sum_{j=1}^{C} e^{z_j - c}\right)$$

#### How to read it out loud:

> *"The stable softmax of z_i equals e raised to the power of z_i minus the maximum of z, divided by the sum of e raised to each z_j minus the maximum of z."*

#### Breakdown of every piece:

* **$z_i$**: The raw logit score for class $i$.
* **$c = \max(\mathbf{z})$**: The single largest number in the logit vector.
* **$z_i - c$**: Shifted logits. Because $c$ is the max, every shifted term is $\le 0$.
* **$e^{z_i - c}$**: The exponential of a non-positive number is strictly bounded between $0$ and $1$, preventing floating-point overflow.
* **$\sum e^{z_j - c}$**: The normalization denominator, guaranteed to be at least $1.0$ because the maximum term contributes $e^{c - c} = e^0 = 1.0$, preventing division by zero.

#### Why AI Cares:

* **Attention Weights in Transformers:** The core self-attention formula $\text{Softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right)$ uses this exact stabilized operation. Dividing by $\sqrt{d_k}$ also keeps logit variance small before entering the Softmax.
* **Log-Softmax in Cross-Entropy Loss:** Deep learning frameworks never run `Softmax` followed by `log()`. They fuse them into `LogSoftmax` using the Log-Sum-Exp trick to retain gradient precision across training steps.

#### NumPy Implementation

In [13]:
# Define input logits with very large numbers (e.g. from an unscaled attention dot product)
# Evaluating np.exp(1000.0) directly exceeds the 64-bit float limit (~1.79e308) and causes an overflow
logits_extreme = np.array([1000.0, 1001.0, 998.0])


# ==========================================
# Method A: Naive Softmax (Demonstrates the Overflow Failure)
# ==========================================
def naive_softmax(z):
    # Suppress the expected runtime warnings locally so notebook output remains clean
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        
        # Step 1: Directly exponentiate the raw numbers
        # 1000.0 is too large to fit in a standard float, so NumPy turns each into 'inf' (infinity)
        exp_z = np.exp(z)  # Becomes: array([inf, inf, inf])
        
        # Step 2: Sum all exponentials
        # inf + inf + inf = inf
        sum_exp = np.sum(exp_z)  # Becomes: inf
        
        # Step 3: Divide each term by the sum
        # In mathematics and computing, inf / inf is undefined, so NumPy returns NaN (Not a Number)
        probs = exp_z / sum_exp  # Becomes: array([nan, nan, nan])
        
    return probs

probs_naive = naive_softmax(logits_extreme)


# ==========================================
# Method B: Numerically Stable Softmax (The "Subtract Max" Trick)
# ==========================================
def stable_softmax(z):
    # Step 1: Find the largest single number in the vector
    # Here, c = 1001.0
    c = np.max(z)
    
    # Step 2: Subtract this maximum from every element in the array
    # Shifted values: [1000 - 1001, 1001 - 1001, 998 - 1001] = [-1.0, 0.0, -3.0]
    # Because every number is now <= 0, exp(value) will strictly be between 0.0 and 1.0
    # This mathematically guarantees that no number can ever overflow to infinity
    shifted_z = z - c
    exp_shifted = np.exp(shifted_z)  # [e^-1, e^0, e^-3] = [0.3679, 1.0, 0.0498]
    
    # Step 3: Sum the safe exponentials
    # Because the largest term is e^0 = 1.0, the sum is guaranteed to be >= 1.0 (no division by zero)
    sum_exp_shifted = np.sum(exp_shifted)  # 0.3679 + 1.0 + 0.0498 = 1.4177
    
    # Step 4: Divide to compute the final valid probability distribution
    probs = exp_shifted / sum_exp_shifted
    return probs

probs_stable = stable_softmax(logits_extreme)


# ==========================================
# Method C: Numerically Stable Log-Sum-Exp (LSE)
# Computes log(sum(exp(z))) without ever computing large exponentials
# ==========================================
def stable_logsumexp(z):
    # Step 1: Identify maximum logit
    c = np.max(z)
    
    # Step 2: Compute LSE identity: c + log(sum(exp(z - c)))
    # Instead of taking log(inf), we compute log of a small, safe sum (1.4177) and add c back
    lse = c + np.log(np.sum(np.exp(z - c)))
    return lse

# Compute log probabilities directly: log(Softmax(z)) = z - LSE(z)
# This avoids doing Softmax first and then taking np.log(), preventing log(0) errors
log_probs_stable = logits_extreme - stable_logsumexp(logits_extreme)


# ==========================================
# Display Results
# ==========================================
print(f"Extreme Logits Input              : {logits_extreme}\n")
print(f"Method A (Naive Softmax - Broken) : {probs_naive}")
print(f"Method B (Stable Softmax Probabilities) : {np.round(probs_stable, 4)}")
print(f"Verification (Sum of Probabilities)    : {np.sum(probs_stable):.1f}")
print(f"Method C (Stable Log-Softmax Values)   : {np.round(log_probs_stable, 4)}")

Extreme Logits Input              : [1000. 1001.  998.]

Method A (Naive Softmax - Broken) : [nan nan nan]
Method B (Stable Softmax Probabilities) : [0.2595 0.7054 0.0351]
Verification (Sum of Probabilities)    : 1.0
Method C (Stable Log-Softmax Values)   : [-1.349 -0.349 -3.349]


In [14]:
sns.set_theme(style="whitegrid")
palette = sns.color_palette("deep")

def stable_softmax(z):
    c = np.max(z)
    exp_shifted = np.exp(z - c)
    return exp_shifted / np.sum(exp_shifted)

def stable_logsumexp(z):
    c = np.max(z)
    return c + np.log(np.sum(np.exp(z - c)))

@interact(
    shift_offset=FloatSlider(
        min=500.0, max=1100.0, step=50.0, value=1000.0,
        description="Base Scale:",
        continuous_update=True
    )
)
def plot_numerical_stability(shift_offset=1000.0):
    logits = np.array([shift_offset, shift_offset + 1.0, shift_offset - 2.0])
    c = np.max(logits)
    shifted_logits = logits - c
    safe_exp = np.exp(shifted_logits)
    probs = safe_exp / np.sum(safe_exp)
    log_probs = logits - stable_logsumexp(logits)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [1.1, 1]})

    # -------------------------------------------------------------
    # Panel 1: Exponentiation Curve & Shifted Safe Domain
    # -------------------------------------------------------------
    z_domain = np.linspace(-4.0, 4.0, 300)
    exp_domain = np.exp(z_domain)

    axes[0].plot(z_domain, exp_domain, color="black", linewidth=2.5, label=r"$y = e^z$")
    
    # Safe zone: Plain text arrow avoids all mathtext parser symbol errors
    axes[0].axvspan(-4.0, 0.0, color=palette[2], alpha=0.15, label="Safe Zone (z <= 0 -> exp(z) in (0, 1])")
    axes[0].axvline(0.0, color="gray", linestyle="--", linewidth=1.2)

    axes[0].scatter(shifted_logits, safe_exp, color=palette[3], s=90, zorder=6)
    for idx, (sz, sexp) in enumerate(zip(shifted_logits, safe_exp)):
        axes[0].annotate(
            f"z_{idx} - c = {sz:.1f}\nexp({sz:.1f}) = {sexp:.3f}",
            (sz, sexp),
            textcoords="offset points",
            xytext=(-35, 12 if sz != 0 else -28),
            fontsize=9, weight="bold", color=palette[3]
        )

    axes[0].annotate(
        f"Original logits (z ~ {shift_offset:.0f})\n-> exp(1000) >> 1.79e308 (Overflow -> inf)",
        xy=(3.5, np.exp(3.5)), xytext=(0.2, 35),
        arrowprops={"facecolor": "#d62728", "shrink": 0.08, "width": 2, "headwidth": 7},
        fontsize=9.5, weight="bold", color="#d62728"
    )

    axes[0].set_xlim(-4.2, 4.2)
    axes[0].set_ylim(-1.0, 58.0)
    axes[0].set_xlabel("Exponent Input (z vs z - c)", fontsize=11)
    axes[0].set_ylabel("Computed Exponent (exp(z))", fontsize=11)
    axes[0].set_title(
        "The 'Subtract Max' Shift: c = max(z)\nGuarantees inputs <= 0, bounding exp(z-c) in (0, 1]",
        fontsize=11, weight="bold"
    )
    axes[0].legend(loc="upper left", frameon=True, fontsize=9)

    # -------------------------------------------------------------
    # Panel 2: Result Comparison (Naive vs Stable Probabilities)
    # -------------------------------------------------------------
    labels = [f"z0 = {logits[0]:.0f}", f"z1 = {logits[1]:.0f}", f"z2 = {logits[2]:.0f}"]
    x_pos = np.arange(len(labels))
    bar_width = 0.35

    # Naive bars (Overflow failure indicator)
    axes[1].bar(
        x_pos - bar_width/2, [0, 0, 0], width=bar_width,
        color="#d62728", alpha=0.3, edgecolor="#d62728", linewidth=1.5,
        label="Naive: exp(z) -> inf / inf = NaN"
    )
    for i in x_pos:
        axes[1].text(i - bar_width/2, 0.05, "NaN\n(Crash)", ha="center", va="bottom", fontsize=9, weight="bold", color="#d62728")

    bars_stable = axes[1].bar(
        x_pos + bar_width/2, probs, width=bar_width,
        color=palette[0], alpha=0.85, edgecolor="black",
        label="Stable: exp(z - c) / sum(exp(z - c))"
    )

    for i, b in enumerate(bars_stable):
        h = b.get_height()
        axes[1].text(
            b.get_x() + b.get_width()/2, h + 0.03,
            f"p = {h:.4f}\nln(p) = {log_probs[i]:.2f}",
            ha="center", va="bottom", fontsize=9, weight="bold", color=palette[0]
        )

    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels(labels, fontsize=11, weight="bold")
    axes[1].set_ylim(0, 1.15)
    axes[1].set_ylabel("Normalized Probability", fontsize=11)
    axes[1].set_title(
        "Softmax Output: Method A (NaN) vs Method B (Valid)",
        fontsize=11, weight="bold"
    )
    axes[1].legend(loc="upper right", frameon=True, fontsize=9)

    fig.suptitle(
        f"Numerical Stability Analysis for Large Logits (c = max(z) = {c:.1f})",
        fontsize=12, weight="bold"
    )

    plt.tight_layout()
    plt.show()

interactive(children=(FloatSlider(value=1000.0, description='Base Scale:', max=1100.0, min=500.0, step=50.0), …